[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ersilia-os/ub-cedd-projects-workshop/blob/main/projects/blue/notebooks/blue_sprint_filter.ipynb)

# Filter the shape hits with SPRINT

**Blue group · Cryptosporidiosis**

The shape-similarity notebook left us with roughly two thousand molecules, which is still too many to look at one by one. Here we put them in order using SPRINT, a neural network that scores a molecule against a protein, and keep the best thousand. Read section 5 before you trust the ordering: the score is useful for sorting a long list, but it is not a prediction that a molecule binds CpABC1.

## What you will do

- Load the molecules that came out of the shape-similarity notebook.
- Score each one against CpABC1 with SPRINT.
- Look at what the scores mean, and at what they do not mean.
- Check the filter did not simply pick the biggest molecules, and look at what it did pick.
- Keep the best thousand and save them for the next step.

## Setup

Run the cell below first. In Colab it downloads the workshop repository (including the data) and installs the packages this project needs. It takes about a minute. **Don't change it.**

In [ ]:
PROJECT = "blue"
NEEDS_GPU = False
import os, sys, shutil, subprocess
if "google.colab" in sys.modules:
    repo_dir = "/content/ub-cedd-projects-workshop"
    if not os.path.exists(repo_dir):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/ersilia-os/ub-cedd-projects-workshop.git", repo_dir], check=True)
    else:
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=True)
    os.chdir(f"{repo_dir}/projects/{PROJECT}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
for _cached in [m for m in sys.modules if m == "scripts" or m.startswith("scripts.")]:
    del sys.modules[_cached]  # forget helper modules imported before the pull above
has_gpu = shutil.which("nvidia-smi") is not None and subprocess.run(["nvidia-smi"], capture_output=True).returncode == 0
print(f"Python {sys.version.split()[0]} | GPU: {'yes' if has_gpu else 'no'} | Folder: {os.getcwd()}")
if NEEDS_GPU and not has_gpu:
    print("WARNING: this notebook needs a GPU. Go to Runtime > Change runtime type, choose CPU, and run this cell again.")

## 1. The molecules we are filtering

The shape-similarity notebook saved a file called `sand_filtered_hits.csv`: the molecules
whose 3D shape is closest to silymarin, after removing the ones with unsuitable properties.
That file is the input here.

It is not stored in the repository, because it is produced by you rather than by us. In
Colab the cell below asks you to upload it. If you are running this on your own computer it
is already in `data/downloads/`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

DOWNLOADS = Path("data/downloads")
DOWNLOADS.mkdir(parents=True, exist_ok=True)
HITS = DOWNLOADS / "sand_filtered_hits.csv"

if "google.colab" in sys.modules and not HITS.exists():
    from google.colab import files
    print("Upload sand_filtered_hits.csv, saved by the shape-similarity notebook")
    for name in files.upload():
        Path(name).rename(HITS)

hits = pd.read_csv(HITS)
print(f"{len(hits)} molecules to filter")
hits.head()

## 2. How the score works

SPRINT turns a protein into a list of 1024 numbers, and a molecule into its own list of
1024 numbers, in such a way that the two lists point in a similar direction when the
molecule is likely to interact with the protein. The score is the cosine of the angle
between them: 1 means the same direction, 0 means unrelated, and negative means opposite.

Turning the *protein* into numbers is the slow part. It needs a 2.4 GB language model of
protein sequences, which would take most of this session to download. But CpABC1 never
changes, so we did that once and saved the result. That leaves only the molecule side to
compute here, which is quick and needs no GPU.

In [ ]:
import numpy as np
from scripts import sprint_score

projector = sprint_score.load_projector("data/sprint_drug_projector.pt")
target = np.load("data/cpabc1_sprint_embedding.npy")

print(f"CpABC1 is described by {target.shape[0]} numbers")

## 3. Score the molecules

Now every molecule gets a score against CpABC1. Each one is first turned into a Morgan
fingerprint, which records which small fragments the molecule contains, and the fingerprint
goes through the network.

There is nothing random in this, so running it again gives exactly the same numbers. A
molecule that RDKit cannot read comes back as `NaN` and is reported rather than quietly
dropped.

In [ ]:
hits["sprint"] = sprint_score.score_molecules(hits["smiles"], projector, target)

unreadable = int(hits["sprint"].isna().sum())
if unreadable:
    print(f"{unreadable} molecules could not be read and will be left out")
hits = hits.dropna(subset=["sprint"]).reset_index(drop=True)

hits["sprint"].describe().round(3).to_frame().T

## 4. What the scores look like

A histogram shows how the scores are spread. What matters for filtering is that they are
spread out at all: if every molecule scored the same, the ranking would carry no
information and choosing a thousand of them would be arbitrary.

In [ ]:
import stylia

figure, axes = stylia.create_figure(1, 1)
ax = axes.next()
ax.hist(hits["sprint"], bins=60, color=stylia.NamedColors().cobalt)
stylia.label(ax, xlabel="SPRINT score against CpABC1", ylabel="Number of molecules")
figure.tight_layout()

## 5. What this score does and does not mean

This is the most important section in the notebook.

Before using SPRINT we tested it. We built a panel of twenty compounds: seven drugs known
to block human P-glycoprotein, the best-studied relative of CpABC1; eight decoys with no
such activity, deliberately chosen to have the same molecular weights and greasiness as
those drugs; and the five silymarin-family flavonoids. SPRINT ranked the seven real
inhibitors above the decoys almost perfectly.

Then we scored the same twenty compounds against **human carbonic anhydrase II**, a small
enzyme that has nothing whatsoever to do with ABC transporters. It separated the inhibitors
from the decoys just as well. In other words, most of what this score responds to is the
molecule itself, not CpABC1.

> **Note:** So treat the score as a sensible way to put a long list in order, not as
> evidence that a molecule binds CpABC1. The thousand molecules you keep below are a
> reasonable thousand to carry forward, and nothing more than that.

> **Exercise:** How would you test whether a scoring tool really uses the protein you give
> it? The test above is one answer. Can you think of another, and what would you need in
> order to run it?

## 6. Keep the best thousand

We sort by score and keep the top thousand. Any cut-off is a compromise: too strict and a
good molecule is lost, too loose and the next step has too much work to do. A thousand is
small enough to handle and large enough that a few mistakes in the ordering do not matter.

In [ ]:
KEEP = 1000

ranked = hits.sort_values("sprint", ascending=False).reset_index(drop=True)
shortlist = ranked.head(KEEP).copy()

cutoff = shortlist["sprint"].min()
print(f"kept {len(shortlist)} of {len(ranked)}, scores {cutoff:.3f} and above")
shortlist.head()

## 7. Did the filter just pick the biggest molecules?

A scoring tool that quietly prefers large, greasy molecules will look like it is working
while really just measuring size. Comparing the molecules we kept against the ones we
dropped shows whether that happened here.

If the two columns are close, the filter chose on something other than bulk.

In [ ]:
from scripts import chemspace

properties = chemspace.describe(ranked["smiles"])
kept = ranked.index.isin(shortlist.index)

pd.DataFrame({
    "kept": properties[kept].median(),
    "dropped": properties[~kept].median(),
}).round(1)

The same comparison as a picture, one panel per property, with silymarin marked as a
line for reference. Heavy overlap between the kept and dropped distributions is what we
want to see.

In [ ]:
from scripts import shape

seed = chemspace.describe(pd.read_csv("data/silymarin.csv")["smiles"])

figure, axes = stylia.create_figure(2, 3, width=1.0)
shape.plot_properties(axes, properties[kept], properties[~kept], seed)
figure.tight_layout()

## 8. The molecules that scored highest

Numbers are easier to trust once you have looked at what they picked. Here are the twelve
top-scoring molecules, drawn in 2D with their scores underneath.

Look at them next to silymarin, which is drawn first. Do they share anything with it?

In [ ]:
from scripts import shape

best = shortlist.head(12)
seed_smiles = pd.read_csv("data/silymarin.csv")["smiles"].iloc[0]

shape.draw_molecules(
    [seed_smiles] + best["smiles"].tolist(),
    ["silymarin (the seed)"] + [f"{row.molport_id}\nSPRINT {row.sprint:.2f}"
                                for row in best.itertuples()])

## 9. Does this filter agree with the shape filter?

The previous notebook chose molecules for being *shaped* like silymarin. This one chose
them for scoring well with SPRINT. If the two measure the same thing, chaining them changes
little; if they disagree, the second filter is undoing the work of the first.

We can test that cheaply. Tanimoto similarity compares which fragments two molecules have
in common, so it is a quick stand-in for "how silymarin-like is this molecule". Plotting it
against the SPRINT score shows whether the two pull in the same direction.

In [ ]:
import numpy as np

ranked_fps = shape.fingerprints(ranked["smiles"])
seed_fp = shape.fingerprints([seed_smiles])[0]
ranked["tanimoto"] = shape.tanimoto_similarity(ranked_fps, seed_fp)

correlation = np.corrcoef(ranked["tanimoto"], ranked["sprint"])[0, 1]
print(f"correlation between silymarin-likeness and SPRINT score: {correlation:+.3f}")

Each point below is one molecule. The dashed line is the cut-off: everything above it
was kept.

In [ ]:
figure, axes = stylia.create_figure(1, 1)
ax = axes.next()
ax.scatter(ranked["tanimoto"], ranked["sprint"], s=6, alpha=0.3,
           color=stylia.NamedColors().cobalt)
ax.axhline(cutoff, linestyle="--", linewidth=1, color=stylia.NamedColors().crimson)
stylia.label(ax, xlabel="Tanimoto similarity to silymarin", ylabel="SPRINT score",
             title=f"correlation {correlation:+.2f}")
figure.tight_layout()

> **Note:** A correlation near zero means the two filters are judging different things,
> so this step is genuinely narrowing the list rather than repeating the last one. A clearly
> negative correlation would be a warning: it would mean SPRINT prefers molecules *unlike*
> silymarin, and that keeping its top thousand quietly discards the chemistry the
> pharmacophore work pointed at.

> **Exercise:** Look at the sign and size of the correlation you got, and at the structures
> in section 7. Would you keep the top thousand by SPRINT score, or would you rather keep
> molecules that do well on both measures? Try changing the cut-off below and see how many
> molecules survive both.

## 10. Save the shortlist

The shortlist goes into `outputs/`, which is not stored in the repository, and Colab
downloads it to your computer. Keep both files: the ranked list records the score of every
molecule, which you will want if you later decide a thousand was the wrong number.

In [ ]:
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

ranked[["molport_id", "smiles", "sprint", "tanimoto"]].to_csv(OUT / "sprint_ranked.csv", index=False)
shortlist[["molport_id", "smiles", "sprint"]].to_csv(OUT / "sprint_shortlist_1000.csv", index=False)

if "google.colab" in sys.modules:
    from google.colab import files
    files.download(str(OUT / "sprint_ranked.csv"))
    files.download(str(OUT / "sprint_shortlist_1000.csv"))
print(f"saved {len(ranked)} ranked and {len(shortlist)} shortlisted molecules")

## Summary

- Scored the shape-similarity hits against CpABC1 with SPRINT, using a protein description
  we prepared in advance so that nothing large had to be downloaded.
- Kept the thousand highest-scoring molecules and checked the filter had not simply
  selected the largest ones.
- Saw that the same score separates known transporter inhibitors from decoys even for an
  unrelated enzyme, which is why this is an ordering and not a binding prediction.

**Next:** take `sprint_shortlist_1000.csv` into docking, where the molecules are placed in
the CpABC1 pocket one by one and scored on how well they actually fit.